In [0]:
import sys
import importlib
import pandas as pd

SRC_PATH = "/Workspace/Users/ariamostajeran99@gmail.com/stock_project_V2/stock-mlops-databricks/src"
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

import signal_generator
import config

importlib.reload(signal_generator)
importlib.reload(config)

from signal_generator import SignalGenerator
from config import MODEL_PREDICTIONS_TABLE_NAME

In [0]:
test_predictions_df = spark.table(MODEL_PREDICTIONS_TABLE_NAME).toPandas()
print(test_predictions_df.columns.tolist())
test_predictions_df.head()

In [0]:
up_thresholds = {
    "AAPL": 0.60,
    "MSFT": 0.60,
    "TSLA": 0.70,
}

down_thresholds = {
    "AAPL": 0.40,
    "MSFT": 0.40,
    "TSLA": 0.30,
}

In [0]:
generator = SignalGenerator(
    up_thresholds=up_thresholds,
    down_thresholds=down_thresholds,
    default_up_threshold=0.60,
    default_down_threshold=0.40
)

signals_df = generator.generate_signals(test_predictions_df)
signal_summary_df = generator.summarize_signals(signals_df)
latest_signals_df = generator.latest_signals(signals_df)

signal_summary_df


In [0]:
latest_signals_df

In [0]:
spark.createDataFrame(signals_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("p2_signals")

spark.createDataFrame(signal_summary_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("p2_signal_summary")

spark.createDataFrame(latest_signals_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("p2_latest_signals")